# Naključni gozdovi — B: vaja (telekom)
**Akademija UI za poslovne aplikacije — Modul 2, Nadzorovano učenje v praksi**

Predavatelj je v zvezku `A_nakljucni_gozdovi` pokazal, kako gozd nastane. Zdaj pogledate vanj sami.

### Ponovitev
- **Eno drevo je nestabilno:** če se podatki malo spremenijo, drevo postavi druga vprašanja.
- **Naključni gozd** je veliko različnih dreves. Napoved gozda je povprečje napovedi dreves.
- **Od kod razlike med drevesi:**
  1. vsako drevo se uči na **naključnem vzorcu vrstic** (z vračanjem),
  2. pri vsakem vprašanju sme izbirati samo med **naključnim delom značilk**.
- **Verjetnost gozda** je povprečje verjetnosti posameznih dreves.
- **Pridobimo:** boljše in bolj stabilne napovedi. **Izgubimo:** enega samega drevesa ne moremo več prebrati.
- **Glavne nastavitve:** `n_estimators` (število dreves), `min_samples_leaf`, `max_features`.

### Nastavitev okolja
Zaženite spodnji dve celici (**Shift + Enter**). Ničesar ne spreminjajte. Koda prve celice je skrita.
Druga celica pokliče skupno pripravo podatkov iz `utils.modeliranje`. Veljavne pripravljene tabele ponovno uporabi.

In [ ]:
#@title Nastavitev okolja (kliknite za prikaz kode)
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/valira-ai/ui-za-poslovne-aplikacije.git"

if "google.colab" in sys.modules:
    KOREN = Path("/content/ui-za-poslovne-aplikacije")
    if not (KOREN / "utils/orodja.py").is_file():
        if KOREN.exists():
            raise FileNotFoundError(f"Mapa {KOREN} že obstaja, vendar v njej ni utils/orodja.py.")
        subprocess.run(["git", "clone", "-q", REPO_URL, str(KOREN)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(KOREN / "requirements.txt")],
        check=True,
    )
else:
    KOREN = Path.cwd().resolve()
    while not (KOREN / "utils/orodja.py").is_file() and KOREN != KOREN.parent:
        KOREN = KOREN.parent
    if not (KOREN / "utils/orodja.py").is_file():
        raise FileNotFoundError("Zaženite zvezek v prenesenem repozitoriju ali njegovi podmapi (manjka utils/orodja.py).")

if str(KOREN) not in sys.path:
    sys.path.insert(0, str(KOREN))
print("Gradivo je v:", KOREN)


In [ ]:
from utils.modeliranje import poskrbi_za_modeliranje, preberi_shemo, preberi_tabelo
PODATKI = poskrbi_za_modeliranje("telekom")
shema = preberi_shemo(PODATKI / "shema.json")


### Knjižnice in podatki

In [ ]:
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from plotnine import *
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

from utils.orodja import nastavi_slog, MODRA, ORANZNA, ZELENA, SIVA, SEME, TELEKOM_STEVILSKE
from utils.orodja import narisi_drevo, vrstica_lestvice
nastavi_slog()

<details>
<summary><b>Kako so pripravljeni podatki? (klikni)</b></summary>

Podatke je pripravila skripta `scripts/pripravi_telekom.py`. Naredi isto kot zvezka predavatelja
`A1_Priprava_Telekom` (mapa `03_Uvod_v_strojno_ucenje/notebooks/01_priprava_podatkov`) in
`A1_Delitev_in_manjkajoce_Telekom` (mapa `03_Uvod_v_strojno_ucenje/notebooks/02_preverjanje_modela`).
Tam so vse odločitve z razlogi. Na kratko:

1. Ena vrstica = ena stranka (100 000 strank). Cilj je `odhod`. `id_stranke` ni značilka.
2. Tabela ima 64 značilk: 58 številskih in 6 besedilnih. Stolpcem iz 1. modula so dodani novi
   (3- in 6-mesečna povprečja, razmerja, logaritmi).
3. Skrite oznake (`U`, `UNKW`) in nemogoče negativne vrednosti so postale manjkajoče vrednosti.
4. Delitev 60 / 20 / 20: naključna, z enakim deležem odhodov, seme 42. Vsi imamo iste stranke v istih množicah.
5. Manjkajoče številke so zapolnjene z **mediano učne množice**. Stolpci `..._manjka` povedo, kje je vrednost manjkala.
6. Besedilni stolpci so zapisani tudi kot stolpci 0/1 (na primer `regija_CHICAGO_AREA`).
   Skupaj je 96 stolpcev, ki jih lahko podamo modelu.
7. Osamelcev ne odstranimo in značilk ne skaliramo: drevo vpraša samo, ali je vrednost nad ali pod pragom.
8. V prikazu uporabimo 16 številskih značilk iz 1. modula, da so modeli na lestvici primerljivi.
9. Modele primerjamo na validacijski množici. Testna ostane zaprta do tekmovanja na koncu klasifikacijskega dela.

</details>

In [ ]:
ucni = preberi_tabelo(PODATKI / "train.csv", shema)
validacijski = preberi_tabelo(PODATKI / "val.csv", shema)

ZNACILKE = TELEKOM_STEVILSKE                                           # 16 številskih značilk iz 1. modula
X_ucni, y_ucni = ucni[ZNACILKE], ucni["odhod"]
X_val, y_val = validacijski[ZNACILKE], validacijski["odhod"]
print(f"Učnih strank: {len(ucni)}, validacijskih: {len(validacijski)}, od teh je odšlo {y_val.mean():.1%}")
print(f"Stolpcev v tabeli: {ucni.shape[1]}, manjkajočih vrednosti: {int(ucni.isna().sum().sum())}")

---
## B1. Poglejte v gozd
Naučimo gozd iz 100 dreves in narišemo vrh enega od njih (prvi dve vprašanji).
Zamenjajte `STEVILKA_DREVESA` (od 0 do 99).

**Vprašanje:** Ali vsa drevesa začnejo z istim vprašanjem?

In [ ]:
gozd = RandomForestClassifier(n_estimators=100, min_samples_leaf=20, random_state=SEME, n_jobs=-1).fit(X_ucni, y_ucni)
print(f"Gozd: validacijska točnost {gozd.score(X_val, y_val):.3f}")

In [ ]:
STEVILKA_DREVESA = [2]  # ← zamenjajte ali dodajte: 0 … 99

for i in STEVILKA_DREVESA:
    drevo = gozd.estimators_[i]
    narisi_drevo(drevo, ZNACILKE, globina=2)
    plt.title(f"Drevo {i} (prikazani sta prvi dve ravni)")
    plt.show()
    print(f"Točnost tega drevesa: {(drevo.predict(X_val.values) == y_val.values).mean():.3f}   točnost gozda: {gozd.score(X_val, y_val):.3f}")

<details>
<summary><b>Rešitev (klikni)</b></summary>

Ne. V tem gozdu 26 dreves začne z `dni_z_napravo`, 17 s `cena_naprave`, 17 z `mesecev_narocnine`, ostala z drugimi značilkami.
Prva vprašanja se razlikujejo zaradi dveh virov naključja: drevesa dobijo različne učne vzorce,
pri vsakem razcepu pa vidijo samo naključen del značilk. Zato ista značilka ni vedno na voljo ali najboljša.

</details>

Preverimo: prvo vprašanje vseh 100 dreves.

In [ ]:
prva_vprasanja = pd.Series([ZNACILKE[drevo.tree_.feature[0]] for drevo in gozd.estimators_])
prva_vprasanja.value_counts()

## B2. Naloga v paru: drugo naključje, drugi stolpci
Na vrhu celice so nastavitve gozda. V paru jih nastavita različno, celico zaženeta in primerjata rezultate.
Spodnja tabela pokaže, katere skupine stolpcev so na voljo (`skupine["ime_skupine"]` je seznam imen stolpcev).

**1. korak: samo drugo seme.** Stolpci ostanejo `TELEKOM_STEVILSKE` (16 stolpcev iz 1. modula).
**Levi v paru** nastavi `SEME_GOZDA = 1`, **desni v paru** `SEME_GOZDA = 2`.

**2. korak: drugi stolpci.** Seme pustita, zamenjata `MOJE_ZNACILKE`:
- **levi:** `skupine["poraba_in_klici"] + skupine["prihodki"]` (kako stranka telefon uporablja in koliko plača),
- **desni:** `skupine["naprava"] + skupine["narocnisko_razmerje"]` (naprava, trajanje naročnine, regija).

**3. korak: vsi stolpci.** Oba nastavita `MOJE_ZNACILKE = shema["znacilke_za_model"]`.

| | stolpcev | prvo vprašanje drevesa 0 | točnost drevesa 0 | točnost gozda | odšlo od 100 |
|---|---|---|---|---|---|
| 1. korak, seme 1 | 16 | | | | |
| 1. korak, seme 2 | 16 | | | | |
| 2. korak, levi | | | | | |
| 2. korak, desni | | | | | |
| 3. korak | | | | | |

**Vprašanja:**
1. Prvi korak: kaj je pri vama različno in kaj podobno? Kaj to pove o enem drevesu in o gozdu?
2. Drugi korak: s katero skupino stolpcev je gozd točnejši? Je kateri boljši od gozda s 16 stolpci?
3. Tretji korak: koliko pridobimo, ko gozd dobi vse stolpce namesto 16?

In [ ]:
shema = preberi_shemo(PODATKI / "shema.json")      # opis tabele, ki ga je shranila skripta
skupine = shema["skupine_za_model"]                                            # ime skupine -> seznam imen stolpcev

print("Vseh stolpcev, ki jih lahko podamo modelu:", len(shema["znacilke_za_model"]))
print(pd.DataFrame([{"skupina": ime, "stolpcev": len(stolpci), "prvi trije stolpci": ", ".join(stolpci[:3])}
                    for ime, stolpci in skupine.items()]).to_string(index=False))

In [ ]:
MOJE_ZNACILKE = TELEKOM_STEVILSKE     # ← stolpci, ki jih gozd sme uporabiti (seznam imen stolpcev)
SEME_GOZDA = 1                        # ← seme naključja (random_state)
STEVILO_DREVES = 100                  # število dreves (n_estimators)
NAJMANJ_V_LISTU = 20                  # najmanj strank v listu (min_samples_leaf)

moj_gozd = RandomForestClassifier(n_estimators=STEVILO_DREVES, min_samples_leaf=NAJMANJ_V_LISTU,
                                  random_state=SEME_GOZDA, n_jobs=-1).fit(ucni[MOJE_ZNACILKE], y_ucni)
drevo_0 = moj_gozd.estimators_[0]
verjetnosti = pd.Series(moj_gozd.predict_proba(validacijski[MOJE_ZNACILKE])[:, 1], index=validacijski.index)

narisi_drevo(drevo_0, list(MOJE_ZNACILKE), globina=2, velikost=(11, 4.5))
plt.title(f"Seme {SEME_GOZDA}, {len(MOJE_ZNACILKE)} stolpcev: drevo 0")
plt.show()
print(f"Stolpcev:                 {len(MOJE_ZNACILKE)}")
print(f"Prvo vprašanje drevesa 0: {MOJE_ZNACILKE[drevo_0.tree_.feature[0]]} <= {drevo_0.tree_.threshold[0]:.2f}")
print(f"Točnost drevesa 0:        {(drevo_0.predict(validacijski[MOJE_ZNACILKE].values) == y_val.values).mean():.3f}")
print(f"Točnost gozda:            {moj_gozd.score(validacijski[MOJE_ZNACILKE], y_val):.3f}")
print(f"Lestvica:                 {vrstica_lestvice('', verjetnosti, y_val)}")

<details>
<summary><b>Rešitev (klikni)</b></summary>

| | stolpcev | prvo vprašanje drevesa 0 | točnost drevesa 0 | točnost gozda | odšlo od 100 |
|---|---|---|---|---|---|
| 1. korak, seme 1 | 16 | `cena_naprave <= 69,99` | 0,561 | 0,621 | 85 |
| 1. korak, seme 2 | 16 | `povp_mesecna_narocnina <= 34,99` | 0,566 | 0,619 | 86 |
| 2. korak, levi (poraba in prihodki, seme 1) | 23 | `sprememba_minut <= 0,62` | 0,529 | 0,583 | 91 |
| 2. korak, desni (naprava in naročniško razmerje, seme 2) | 37 | `mesecev_narocnine <= 10,50` | 0,579 | 0,596 | 73 |
| 3. korak, seme 1 | 96 | `povp_minute <= 0,38` | 0,556 | 0,622 | 87 |
| 3. korak, seme 2 | 96 | `log1p_cena_naprave <= 4,50` | 0,554 | 0,626 | 89 |

1. Posamezna drevesa se razlikujejo, ker dobijo različne vzorce strank in izbore značilk.
   Točnost celotnega gozda na **vseh validacijskih strankah** je pri dveh semenih podobna (0,621 in 0,619).
   Povprečenje lahko zmanjša nihanje posameznih dreves; ta dva rezultata pa sama še ne dokazujeta, da bo vsak gozd stabilen.
2. Gozd z napravo in naročniškim razmerjem pravilno razvrsti več validacijskih strank (točnost 0,596 proti 0,583).
   Oba zaostajata za gozdom s 16 stolpci (0,621 in 0,619); iz te primerjave same ne vemo, kateri posamezni stolpci povzročijo razliko.
   Če pa izberemo samo **100 strank z najvišjo napovedano verjetnostjo**, gozd s porabo in prihodki najde več dejanskih odhodov (91 proti 73).
   Točnost na vseh strankah in kakovost prvih 100 sta različni merili.
3. Pri teh dveh semenih vseh 96 stolpcev da validacijski točnosti 0,622 in 0,626, osnovnih 16 pa 0,621 in 0,619.
   Razlika je manjša od ene odstotne točke. Dodatni stolpci v tem poskusu pomagajo le malo; več stolpcev ni zagotovilo za boljši model.

</details>

Preverimo vse korake naenkrat:

In [ ]:
poskusi = [("1. korak, seme 1", TELEKOM_STEVILSKE, 1),
           ("1. korak, seme 2", TELEKOM_STEVILSKE, 2),
           ("2. korak, levi: poraba in prihodki", skupine["poraba_in_klici"] + skupine["prihodki"], 1),
           ("2. korak, desni: naprava in naročniško razmerje", skupine["naprava"] + skupine["narocnisko_razmerje"], 2),
           ("3. korak, seme 1", shema["znacilke_za_model"], 1),
           ("3. korak, seme 2", shema["znacilke_za_model"], 2)]
vrstice = []
for ime, znacilke, seme in poskusi:
    g = RandomForestClassifier(n_estimators=100, min_samples_leaf=20, random_state=seme, n_jobs=-1).fit(ucni[znacilke], y_ucni)
    d0 = g.estimators_[0]
    v = pd.Series(g.predict_proba(validacijski[znacilke])[:, 1], index=validacijski.index)
    vrstice.append({"poskus": ime, "stolpcev": len(znacilke),
                    "prvo vprašanje drevesa 0": f"{znacilke[d0.tree_.feature[0]]} <= {d0.tree_.threshold[0]:.2f}",
                    "točnost drevesa 0": f"{(d0.predict(validacijski[znacilke].values) == y_val.values).mean():.3f}",
                    "točnost gozda": f"{g.score(validacijski[znacilke], y_val):.3f}",
                    "odšlo od 100": vrstica_lestvice("", v, y_val)["odšlo od 100"]})
print(pd.DataFrame(vrstice).to_string(index=False))

## B3. Koliko dreves?
Poskusite `STEVILO_DREVES = 1, 5, 20, 100` in si zapišite validacijsko točnost.

**Vprašanje:** Kje se točnost neha izboljševati?

In [ ]:
STEVILO_DREVES = 5          # ← zamenjajte

g = RandomForestClassifier(n_estimators=STEVILO_DREVES, min_samples_leaf=20, random_state=SEME, n_jobs=-1).fit(X_ucni, y_ucni)
print(f"{STEVILO_DREVES} dreves: validacijska točnost {g.score(X_val, y_val):.3f}")

<details>
<summary><b>Rešitev (klikni)</b></summary>

Na **isti validacijski množici** se točnost poveča z 0,563 pri enem drevesu na 0,614 pri desetih,
0,619 pri tridesetih in 0,621 pri stotih. Največji prirastek je na začetku; med 30 in 100 drevesi je le 0,002.
To velja za te nastavitve in to seme.

</details>

Preverimo:

In [ ]:
for stevilo in [1, 5, 10, 20, 30, 100]:
    g = RandomForestClassifier(n_estimators=stevilo, min_samples_leaf=20, random_state=SEME, n_jobs=-1).fit(X_ucni, y_ucni)
    print(f"{stevilo:3d} dreves: validacijska točnost {g.score(X_val, y_val):.3f}")

## B4. Ena stranka, 100 mnenj
Izberite stranko (`STRANKA` je zaporedna številka v validacijski množici). Graf pokaže, kakšno verjetnost odhoda ji da
vsako od 100 dreves. Oranžna črta je povprečje — to je verjetnost, ki jo vrne gozd.

**Vprašanje:** Poiščite stranko, pri kateri so si drevesa zelo enotna, in stranko, pri kateri niso.

In [ ]:
STRANKA = 0          # ← zamenjajte: 0 … 19999

stranka = X_val.iloc[[STRANKA]]
mnenja = pd.DataFrame({"verjetnost": [drevo.predict_proba(stranka.values)[0, 1] for drevo in gozd.estimators_]})
print(f"Gozd: verjetnost odhoda {gozd.predict_proba(stranka)[0, 1]:.2f}.   Dejansko: {'odšla' if y_val.iloc[STRANKA] == 1 else 'ostala'}.")

(ggplot(mnenja, aes(x="verjetnost"))
 + geom_histogram(bins=20, fill=MODRA)
 + geom_vline(xintercept=mnenja["verjetnost"].mean(), color=ORANZNA, size=1.5)
 + scale_x_continuous(limits=(-0.03, 1.03))
 + labs(title=f"Stranka {STRANKA}: mnenja 100 dreves", x="verjetnost odhoda po mnenju posameznega drevesa", y="število dreves"))

---
### Koliko značilk sme videti drevo?
`max_features` pove, med koliko naključnimi značilkami drevo izbira pri vsakem vprašanju.
Poskusite `1` (drevesa so si najbolj različna), `"sqrt"` (privzeto: 4 od 16) in `None` (vsa drevesa vidijo vse značilke).

In [ ]:
MAX_FEATURES = "sqrt"          # ← zamenjajte: 1, "sqrt", None

g = RandomForestClassifier(n_estimators=100, min_samples_leaf=20, max_features=MAX_FEATURES, random_state=SEME, n_jobs=-1).fit(X_ucni, y_ucni)
razlicnih = len(set(drevo.tree_.feature[0] for drevo in g.estimators_))
print(f"max_features={MAX_FEATURES}: validacijska točnost {g.score(X_val, y_val):.3f}, različnih prvih vprašanj: {razlicnih}")

<details>
<summary><b>Rešitev (klikni)</b></summary>

- `1`: točnost 0,612, 16 različnih prvih vprašanj. Drevesa so zelo različna, a vsako posebej je šibko.
- `"sqrt"`: točnost 0,621, 10 različnih prvih vprašanj.
- `None`: točnost 0,617, eno samo prvo vprašanje. V tem poskusu imajo vsa drevesa enako prvo značilko,
  čeprav se zaradi različnih vzorcev učnih strank lahko razlikujejo nižje v drevesu.

Med temi tremi izbirami ima `"sqrt"` najvišjo validacijsko točnost (0,621). Naključni izbor značilk
poveča raznolikost dreves, vendar sam po sebi ne zagotovi boljše napovedi.

</details>

Preverimo:

In [ ]:
for mf in [1, "sqrt", None]:
    g = RandomForestClassifier(n_estimators=100, min_samples_leaf=20, max_features=mf, random_state=SEME, n_jobs=-1).fit(X_ucni, y_ucni)
    razlicnih = len(set(drevo.tree_.feature[0] for drevo in g.estimators_))
    print(f"max_features={str(mf):5s}: validacijska točnost {g.score(X_val, y_val):.3f}, različnih prvih vprašanj: {razlicnih}")

---
## B5. Moj kandidat za tekmovanje
Na koncu klasifikacijskega dela bomo modele vseh udeležencev primerjali na **testni množici**.
Tu pripravite svojega kandidata: izberite stolpce, nastavitve gozda in seme. Celica izpiše validacijsko točnost,
vrstico lestvice in opis kandidata. Opis si shranite (kopirajte izpis).

Testne množice tukaj ne odpremo: odpre jo šele zvezek za tekmovanje.

Namigi: stolpce izberite s tabelo skupin iz B2, na primer `skupine["naprava"] + skupine["prihodki"]`,
ali naštejte imena: `["dni_z_napravo", "mesecev_narocnine", "povp_minute"]`. Nastavitve poznate iz B3 in iz dela "Koliko značilk sme videti drevo?".

In [ ]:
MOJE_ZNACILKE = TELEKOM_STEVILSKE     # ← stolpci vašega kandidata (seznam imen stolpcev)
STEVILO_DREVES = 100                  # ← število dreves (n_estimators)
NAJMANJ_V_LISTU = 20                  # ← najmanj strank v listu (min_samples_leaf)
MAX_FEATURES = "sqrt"                 # ← koliko značilk vidi drevo pri vsakem vprašanju (max_features)
SEME_GOZDA = 42                       # ← seme naključja (random_state)

kandidat_gozd = RandomForestClassifier(n_estimators=STEVILO_DREVES, min_samples_leaf=NAJMANJ_V_LISTU,
                                       max_features=MAX_FEATURES, random_state=SEME_GOZDA,
                                       n_jobs=-1).fit(ucni[MOJE_ZNACILKE], y_ucni)
tocnost = kandidat_gozd.score(validacijski[MOJE_ZNACILKE], y_val)
verjetnosti = pd.Series(kandidat_gozd.predict_proba(validacijski[MOJE_ZNACILKE])[:, 1], index=validacijski.index)
print(f"Validacijska točnost: {tocnost:.4f}   (gozd iz B1: {gozd.score(X_val, y_val):.4f})")
print(f"Lestvica:             {vrstica_lestvice('', verjetnosti, y_val)}")

kandidat = {"model": "gozd",
            "znacilke": list(MOJE_ZNACILKE),
            "nastavitve": {"n_estimators": STEVILO_DREVES, "min_samples_leaf": NAJMANJ_V_LISTU, "max_features": MAX_FEATURES},
            "seme": SEME_GOZDA,
            "tocnost_validacija": round(tocnost, 4)}
print()
print(kandidat)